In [2]:
# Broadway Grosses ANN Model

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

# ============================================================
# Load and clean dataset
# ============================================================

data = pd.read_csv('broadway.csv')

# Remove rows with missing values
# (You can also use fillna() if preferred)
data = data.dropna()

# ============================================================
# Choose target variable
# ============================================================
# Predicting average ticket price
# You can change this to another column if desired

y = data['Average_ticket']

# Features
X = data.drop(columns=['Average_ticket'])

# ============================================================
# One-hot encode categorical columns
# ============================================================

categorical_cols = ['Show_name', 'Year']

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
).astype('float32')

# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================================
# Standardization
# ============================================================

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ============================================================
# Model architecture
# ============================================================

model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.1),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.1),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(32, activation='relu'),

    # Regression output
    layers.Dense(1)
])

# ============================================================
# Compile model
# ============================================================

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# ============================================================
# Early stopping
# ============================================================

early_stop = keras.callbacks.EarlyStopping(
    patience=50,
    restore_best_weights=True
)

# ============================================================
# Train model
# ============================================================

history = model.fit(
    X_train_s,
    y_train,
    validation_split=0.2,
    epochs=1000,
    batch_size=8,
    callbacks=[early_stop],
    verbose=1
)

# ============================================================
# Evaluate model
# ============================================================

loss, mae = model.evaluate(X_test_s, y_test, verbose=0)
print(f"Test MAE: {mae:.2f}")
print(f"  MSE Loss (scaled)   : {loss:.6f}")

# ============================================================
# Save outputs
# ============================================================

model.save('broadway_model.h5')
np.save('scaler_mean.npy', scaler.mean_)
np.save('scaler_scale.npy', scaler.scale_)

with open('feature_columns.txt', 'w') as f:
    f.write('\n'.join(X.columns))

print('Saved: broadway_model.h5, scaler_mean.npy, scaler_scale.npy, feature_columns.txt')


Epoch 1/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 49s 12ms/step - loss: 546.7847 - mae: 15.5753 - val_loss: 278.5977 - val_mae: 12.2608
Epoch 2/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 44s 11ms/step - loss: 258.0146 - mae: 11.3870 - val_loss: 349.2132 - val_mae: 13.8051
Epoch 3/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 42s 11ms/step - loss: 198.4170 - mae: 9.7908 - val_loss: 464.6957 - val_mae: 17.2551
Epoch 4/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 47s 12ms/step - loss: 172.6218 - mae: 9.0987 - val_loss: 167.9631 - val_mae: 9.4139
Epoch 5/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 42s 11ms/step - loss: 147.1441 - mae: 8.4244 - val_loss: 340.3133 - val_mae: 13.4858
Epoch 6/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 44s 12ms/step - loss: 126.5999 - mae: 7.9145 - val_loss: 449.5152 - val_mae: 16.3347
Epoch 7/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 43s 11ms/step - loss: 120.6128 - mae: 7.5622 - val_loss: 395.5166 - val_mae: 15.7432
Epoch 8/1000
3816/3816 ━━━━━━━━━━━━━━━━━━━━ 42s 11ms/step - loss: 107.5128 - mae: 7.1367 - val_l

Test MAE: 9.56
  MSE Loss (scaled)   : 175.185349
Saved: broadway_model.h5, scaler_mean.npy, scaler_scale.npy, feature_columns.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
